In [1]:
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project="master-trackman-project")
query = """
SELECT GameID, Inning, Top_Bottom, PAofInning, PitchofPA, Batter, PlateLocSide, PlateLocHeight, PitchCall, Balls, Strikes, delta_run_exp
FROM `master-trackman-project.master_trackman_dataset.2025_data`
ORDER BY GameID ASC, Inning ASC, Top_Bottom DESC, PAofInning ASC, PitchofPA ASC
"""
df = client.query(query).to_dataframe()
print(f"Successful Query: {len(df)} pitches")

c:\Users\tyism\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\auth\_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Successful Query: 1501310 pitches


In [2]:
from compute import separate_by_zone

In [3]:
df_zone = separate_by_zone(df)

In [4]:
hitter_count_map = {
        True: '<2k',
        False: '2k'
    }
df_zone['hitter_count'] = (df_zone['Strikes'] < 2).map(hitter_count_map)

In [14]:
df_zone.groupby(['PlateZone','swing_decision', 'hitter_count'])[['delta_run_exp']].mean().reset_index()

,PlateZone,swing_decision,hitter_count,delta_run_exp
0,Chase,swing,2k,-0.106261
1,Chase,swing,<2k,-0.101857
2,Chase,take,2k,0.112312
3,Chase,take,<2k,0.074081
4,Heart,swing,2k,0.050555
5,Heart,swing,<2k,-0.023725
6,Heart,take,2k,-0.277701
7,Heart,take,<2k,-0.105663
8,Shadow,swing,2k,-0.015563
9,Shadow,swing,<2k,-0.069501


In [8]:
query = """
    SELECT *
    FROM `master-trackman-project.dashboard_dataset.weights`
    """
weights = client.query(query).to_dataframe()

In [9]:
weights

,PlateZone,swing_decision,hitter_count,delta_run_exp
0,Chase,swing,2k,-0.106261
1,Chase,swing,<2k,-0.101857
2,Chase,take,2k,0.112312
3,Chase,take,<2k,0.074081
4,Heart,swing,2k,0.050555
5,Heart,swing,<2k,-0.023725
6,Heart,take,2k,-0.277701
7,Heart,take,<2k,-0.105663
8,Shadow,swing,2k,-0.015563
9,Shadow,swing,<2k,-0.069501


In [26]:

client = bigquery.Client(project="master-trackman-project")
query = """
SELECT Batter, PlateLocSide, PlateLocHeight, PitchCall, Balls, Strikes, delta_run_exp, 
FROM `master-trackman-project.master_trackman_dataset.2026_data`
WHERE BatterTeam = 'CSD_TRI'
"""
raw = client.query(query).to_dataframe()
print(f"Successful Query: {len(raw)} pitches")

c:\Users\tyism\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\auth\_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Successful Query: 2584 pitches


In [27]:
raw

,Batter,PlateLocSide,PlateLocHeight,PitchCall,Balls,Strikes,delta_run_exp
0,"Camacho, Gabe",0.81581,1.09163,StrikeSwinging,1,2,-0.927
1,"Martinez, Trevian",-0.03524,3.61876,StrikeSwinging,3,2,-0.720
2,"Costello, Nick",0.03781,1.80768,StrikeCalled,3,2,-0.709
3,"Leopard, Alex",0.36241,3.19785,StrikeSwinging,3,2,-0.579
4,"Potestio, Anthony",-0.58166,2.66265,StrikeCalled,3,2,-0.572
...,...,...,...,...,...,...,...
2579,"Potestio, Anthony",-0.52499,2.56865,InPlay,2,2,0.430
2580,"Camacho, Gabe",-0.04344,1.86311,InPlay,1,0,0.437
2581,"Costello, Nick",-0.10987,1.96741,InPlay,2,1,1.030
2582,"Crossland, Michael",-0.27452,2.34717,InPlay,2,2,1.476


In [28]:
query = """
SELECT *
FROM `master-trackman-project.dashboard_dataset.weights`
"""
weights = client.query(query).to_dataframe()

In [30]:
weights.set_index(['PlateZone','swing_decision', 'hitter_count'])

delta_run_exp
PlateZone swing_decision hitter_count               
Chase     swing          2k                -0.106261
                         <2k               -0.101857
          take           2k                 0.112312
                         <2k                0.074081
Heart     swing          2k                 0.050555
                         <2k               -0.023725
          take           2k                -0.277701
                         <2k               -0.105663
Shadow    swing          2k                -0.015563
                         <2k               -0.069501
          take           2k                -0.039670
                         <2k               -0.024708
Waste     swing          2k                -0.182252
                         <2k               -0.103769
          take           2k                 0.107997
                         <2k                0.085746

In [31]:
## Strike zone
## assuming view is from pitchers mound

lower_boundary = 1.5
upper_boundary = 3.5
left_boundary = -0.833
right_boundary = 0.833

heart_lower_boundary = 1.833
heart_upper_boundary = 3.167
heart_left_boundary = -0.558
heart_right_boundary = 0.558

shadow_lower_boundary = 1.167
shadow_upper_boundary = 3.833
shadow_left_boundary = -1.108
shadow_right_boundary = 1.108

chase_lower_boundary = 0.5
chase_upper_boundary = 4.333
chase_left_boundary = -1.667
chase_right_boundary = 1.667

def separate_by_zone(df):
    zone_df = df.copy()
    zone_df['PlateLocSide'] *= -1
    
    # Default all rows to Waste
    zone_df['PlateZone'] = 'Waste'

    # Heart zone
    mask_heart = (
        (zone_df['PlateLocSide'] >= heart_left_boundary) &
        (zone_df['PlateLocSide'] <= heart_right_boundary) &
        (zone_df['PlateLocHeight'] >= heart_lower_boundary) &
        (zone_df['PlateLocHeight'] <= heart_upper_boundary)
    )
    zone_df.loc[mask_heart, 'PlateZone'] = 'Heart'

    # Shadow zone
    mask_shadow = (
        (zone_df['PlateLocSide'] >= shadow_left_boundary) &
        (zone_df['PlateLocSide'] <= shadow_right_boundary) &
        (zone_df['PlateLocHeight'] >= shadow_lower_boundary) &
        (zone_df['PlateLocHeight'] <= shadow_upper_boundary)
    )
    zone_df.loc[mask_shadow & ~mask_heart, 'PlateZone'] = 'Shadow'

    # Chase zone
    mask_chase = (
        (zone_df['PlateLocSide'] >= chase_left_boundary) &
        (zone_df['PlateLocSide'] <= chase_right_boundary) &
        (zone_df['PlateLocHeight'] >= chase_lower_boundary) &
        (zone_df['PlateLocHeight'] <= chase_upper_boundary)
    )
    zone_df.loc[mask_chase & ~mask_heart & ~mask_shadow, 'PlateZone'] = 'Chase'

    return zone_df

In [34]:
swings = separate_by_zone(raw)
swing_calls = ['StrikeSwinging', 'FoulBallNotFieldable', 'InPlay', 'FoulBallFieldable', 'FoulBallNotFieldable']
swings['swing_decision'] = swings['PitchCall'].apply(lambda x: 'swing' if x in swing_calls else 'take')
hitter_count_map = {
    True: '<2k',
    False: '2k'
}
swings['hitter_count'] = (swings['Strikes'] < 2).map(hitter_count_map)

In [13]:
swings

NameError: name 'swings' is not defined

In [51]:
swing_results = swings.set_index(['PlateZone','swing_decision', 'hitter_count']).drop(columns=['PlateLocSide','PlateLocHeight','PitchCall','Balls','Strikes', 'delta_run_exp']).join(weights.set_index(['PlateZone','swing_decision', 'hitter_count'])).reset_index()
swing_results['swing_decision'] = swing_results['swing_decision'].apply(lambda x: x == 'swing')
swing_results

,PlateZone,swing_decision,hitter_count,Batter,delta_run_exp
0,Chase,True,2k,"Camacho, Gabe",-0.106261
1,Shadow,True,2k,"Martinez, Trevian",-0.015563
2,Shadow,False,2k,"Costello, Nick",-0.039670
3,Shadow,True,2k,"Leopard, Alex",-0.015563
4,Shadow,False,2k,"Potestio, Anthony",-0.039670
...,...,...,...,...,...
2579,Heart,True,2k,"Potestio, Anthony",0.050555
2580,Heart,True,<2k,"Camacho, Gabe",-0.023725
2581,Heart,True,<2k,"Costello, Nick",-0.023725
2582,Heart,True,2k,"Crossland, Michael",0.050555


In [52]:
grouped_columns = ['Batter', 'PlateZone','hitter_count']

swing_results.groupby(grouped_columns)[['delta_run_exp', 'swing_decision']].agg(
    weighted_re_sum=('delta_run_exp', 'sum'),
    swing_percentage=('swing_decision', 'mean'),
    count=('delta_run_exp', 'count')
).reset_index()

,Batter,PlateZone,hitter_count,weighted_re_sum,swing_percentage,count
0,"Allen, J.C.",Chase,2k,0.828536,0.333333,21
1,"Allen, J.C.",Chase,<2k,2.120622,0.180000,50
2,"Allen, J.C.",Heart,2k,0.430070,0.933333,15
3,"Allen, J.C.",Heart,<2k,-2.378640,0.644444,45
4,"Allen, J.C.",Shadow,2k,-0.430263,0.869565,23
...,...,...,...,...,...,...
126,"Widelski, Nathaniel",Chase,<2k,0.490794,0.111111,9
127,"Widelski, Nathaniel",Heart,<2k,-0.258776,0.500000,4
128,"Widelski, Nathaniel",Shadow,2k,-0.110465,0.500000,4
129,"Widelski, Nathaniel",Shadow,<2k,-0.430871,0.250000,12


In [7]:
df.sort_values(['GameID','Inning','Top_Bottom'])

,PitchNo,Date,Time,PAofInning,PitchofPA,Pitcher,PitcherId,PitcherThrows,PitcherTeam,Batter,...,ThrowTrajectoryZc1,ThrowTrajectoryZc2,PitchReleaseConfidence,PitchLocationConfidence,PitchMovementConfidence,HitLaunchConfidence,HitLandingConfidence,CatcherThrowCatchConfidence,CatcherThrowReleaseConfidence,CatcherThrowLocationConfidence
0,113,2025-05-13,19:14:30.320000,2,2,"Brown, Jack",8.153560e+05,Right,LOU_CAR,"Kiper, Christian",...,NaN,NaN,High,High,High,High,High,None,None,None
1,51,2025-04-29,15:13:26.690000,7,3,"Radcliff, Ryan",1.000147e+09,Left,EKU_COL,"Baker, George",...,NaN,NaN,High,High,High,High,Undefined,None,None,None
2,21,2025-02-16,11:19:14.420000,1,3,"Hallam, Isaac",1.014998e+07,Right,SEA_RED,"Wallace, Colby",...,NaN,NaN,High,High,High,None,None,None,None,None
3,193,2025-04-01,20:25:25.440000,1,3,"Crider, Nate",1.000140e+09,Right,LOU_BUL,"Arrambide, Cade",...,59.8745,-3.80402,High,High,High,None,None,High,Medium,Medium
4,341,2025-03-08,16:56:53.170000,2,3,"Gold, Joe",8.223360e+05,Right,BOC_EAG,"Hanson, Luke",...,NaN,NaN,High,High,High,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1501211,157,2025-04-08,19:24:30.620000,2,4,"Hughes, Gavin",1.000211e+09,Right,SIM_UNI,"Ham, Billy",...,NaN,NaN,High,High,High,High,High,None,None,None
1501212,249,2025-03-23,15:37:49.230000,5,4,"LeBlanc, Jacob",1.000410e+09,Left,NOR_DEM,"Dominguez, Adrian",...,NaN,NaN,High,High,High,None,None,None,None,None
1501213,12,2025-04-25,15:50:10.370000,3,2,"MacMillan, Blake",8.323500e+05,Left,NOR_CAT,"DeCarlo, Samuel",...,NaN,NaN,High,High,High,None,None,None,None,None
1501214,243,2025-02-23,15:23:56.170000,4,2,"Smith, Gabe",1.004761e+07,Right,MIS_BEA,"Stevens, Nolan",...,NaN,NaN,High,High,High,None,None,None,None,None


In [45]:
df.sort_values(
    by=['GameID','Inning','Top_Bottom','PAofInning','PitchofPA'],
    ascending=[True, True, False, True, True]
)

,PitchNo,Date,Time,PAofInning,PitchofPA,Pitcher,PitcherId,PitcherThrows,PitcherTeam,Batter,...,ThrowTrajectoryZc1,ThrowTrajectoryZc2,PitchReleaseConfidence,PitchLocationConfidence,PitchMovementConfidence,HitLaunchConfidence,HitLandingConfidence,CatcherThrowCatchConfidence,CatcherThrowReleaseConfidence,CatcherThrowLocationConfidence
0,190,2026-02-17,20:59:57.910000,1,4,"Sears, Merek",8.123730e+05,Left,ARL_MAV,"Liddington, Rob",...,NaN,NaN,High,High,High,None,None,None,None,None
1,196,2026-02-17,21:02:56.500000,4,1,"Sears, Merek",8.123730e+05,Left,ARL_MAV,"Gargett, Kyuss",...,NaN,NaN,High,High,High,None,None,None,None,None
2,197,2026-02-17,21:03:32.070000,4,2,"Sears, Merek",8.123730e+05,Left,ARL_MAV,"Gargett, Kyuss",...,NaN,NaN,High,High,High,None,None,None,None,None
3,204,2026-02-17,21:08:22.570000,1,2,"Nelson, Cade",8.375190e+05,Right,TCU_HFG,"Dygert, Caylon",...,NaN,NaN,High,High,High,None,None,None,None,None
4,227,2026-02-17,21:18:41.030000,1,5,"Sears, Merek",8.123730e+05,Left,ARL_MAV,"Cramer, Cole",...,NaN,NaN,High,High,High,High,Medium,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,262,2026-02-28,18:40:10.250000,3,5,"Rawlings, Eric",1.013235e+07,Right,WIU_LEA,"White, Zachary",...,NaN,NaN,High,High,High,Medium,High,None,None,None
996,249,2026-02-28,18:32:22.870000,5,1,"Hess, Holden",1.000209e+09,Right,ULM_WAR,"McClintock, Josh",...,NaN,NaN,High,High,High,None,None,None,None,None
997,290,2026-02-28,18:56:09.800000,1,1,"Hess, Holden",1.000209e+09,Right,ULM_WAR,"Connolly, Joe",...,NaN,NaN,High,High,High,None,None,None,None,None
998,300,2026-02-28,18:59:58.250000,3,1,"Hess, Holden",1.000209e+09,Right,ULM_WAR,"Monge, Isaiah",...,NaN,NaN,High,High,High,None,None,None,None,None


In [16]:
client = bigquery.Client(project="master-trackman-project")
query = """
SELECT *
FROM `master-trackman-project.dashboard_dataset.swing_results`
"""
swing_results = client.query(query).to_dataframe()
print(f"Successful Query: {len(swing_results)} pitches")

c:\Users\tyism\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\auth\_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Successful Query: 124 pitches


In [18]:
swing_results.groupby(['PlateZone','hitter_count'])[['weighted_re_sum']].agg(lambda x: x.abs().max()).to_dict()

{'weighted_re_sum': {('Chase', '2k'): 1.4842552185495284,
  ('Chase', '<2k'): 4.0374901200273605,
  ('Heart', '2k'): 0.9605470035172237,
  ('Heart', '<2k'): 5.719005393624443,
  ('Shadow', '2k'): 1.1553033714208802,
  ('Shadow', '<2k'): 6.501811178483342,
  ('Waste', '2k'): 2.375937794226499,
  ('Waste', '<2k'): 3.7367548702249547}}

In [ ]:
client = bigquery.Client(project="master-trackman-project")
query = """
SELECT *
FROM `master-trackman-project.dashboard_dataset.weights`
"""
weights = client.query(query).to_dataframe()
print(f"Successful Query: {len(swing_results)} pitches")

c:\Users\tyism\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\auth\_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Successful Query: 124 pitches


In [45]:
weights

,PlateZone,swing_decision,hitter_count,delta_run_exp
0,Chase,swing,2k,-0.106261
1,Chase,swing,<2k,-0.101857
2,Chase,take,2k,0.112312
3,Chase,take,<2k,0.074081
4,Heart,swing,2k,0.050555
5,Heart,swing,<2k,-0.023725
6,Heart,take,2k,-0.277701
7,Heart,take,<2k,-0.105663
8,Shadow,swing,2k,-0.015563
9,Shadow,swing,<2k,-0.069501
